# Tugas Manipulasi Data dengan Python Pandas
**Dataset:** `paket_a_transaksi_pos_2024_2025.csv` (transaksi Point of Sale 2024–2025)

| Task | Topik |
|---|---|
| 1 | Indexing & Selection (pembersihan anomali & imputasi) |
| 2 | Operasi Aritmatika (feature engineering) |
| 3 | Filtering |
| 4 | Sorting |
| 5 | Grouping & Agregasi |
| + | Catatan tambahan: pemeriksaan kualitas data |

In [43]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

## Memuat & Inspeksi Awal Data

In [44]:
df = pd.read_csv('paket_a_transaksi_pos_2024_2025.csv')
print('Ukuran data (baris, kolom):', df.shape)
df.head()

Ukuran data (baris, kolom): (5030, 9)


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000,0,SELESAI
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000,0,SELESAI
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,"7500,00",0,SELESAI
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500,0,SELESAI
4,TRX0243509,26/03/2025,G05,K016,RMT1407,1,6000,0,SELESAI


In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5030 entries, 0 to 5029
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id_transaksi   5030 non-null   object
 1   tanggal        5030 non-null   object
 2   kode_gerai     5030 non-null   object
 3   kode_kasir     4769 non-null   object
 4   kode_produk    5030 non-null   object
 5   jumlah         5030 non-null   int64 
 6   harga_satuan   5030 non-null   object
 7   diskon_persen  5030 non-null   int64 
 8   status         5030 non-null   object
dtypes: int64(2), object(7)
memory usage: 353.8+ KB


In [46]:
# Jumlah nilai kosong per kolom
df.isna().sum()

,0
id_transaksi,0
tanggal,0
kode_gerai,0
kode_kasir,261
kode_produk,0
jumlah,0
harga_satuan,0
diskon_persen,0
status,0


---
# Task 1: Indexing & Selection (Pembersihan Anomali & Imputasi)

## 1.1 Pembersihan format harga (`harga_satuan`)
Kolom `harga_satuan` terbaca sebagai teks (*object/str*) sehingga tidak bisa dihitung. Sebelum dikonversi, kita lihat dulu pola penulisannya (setiap digit diganti `9`).

In [47]:
print('Tipe data harga_satuan:', df['harga_satuan'].dtype)

pola = df['harga_satuan'].str.replace(r'\d', '9', regex=True)
pola.value_counts()

Tipe data harga_satuan: object


,count
harga_satuan,
99999,3974
9999,960
"99999,99",40
99.999,38
"9999,99",11
9.999,7


Ada **tiga** cara penulisan harga di data ini, bukan dua:

| Pola | Contoh | Arti |
|---|---|---|
| angka biasa | `13000` | sudah benar |
| koma desimal | `7500,00` | 7500 (kasus yang disebut di soal) |
| titik pemisah ribuan | `13.000` | 13000 |

Jika hanya mengganti koma menjadi titik seperti di soal, nilai `13.000` akan terbaca sebagai **13,0** (salah 1000×). Karena itu format titik-ribuan ditangani lebih dulu: titik dihapus hanya jika polanya *1–3 digit + titik + tepat 3 digit*. Setelah itu baru koma diganti titik sesuai instruksi soal.

In [48]:
# (a) Titik sebagai pemisah ribuan, mis. '13.000' -> '13000'
mask_ribuan = df['harga_satuan'].str.fullmatch(r'\d{1,3}\.\d{3}')
print('Baris berformat titik-ribuan :', mask_ribuan.sum())
df.loc[mask_ribuan, 'harga_satuan'] = df.loc[mask_ribuan, 'harga_satuan'].str.replace('.', '', regex=False)

# (b) Koma sebagai pemisah desimal, mis. '7500,00' -> '7500.00'
mask_koma = df['harga_satuan'].str.contains(',')
print('Baris berformat koma-desimal :', mask_koma.sum())
df['harga_satuan'] = df['harga_satuan'].str.replace(',', '.', regex=False)

# (c) Konversi ke float
df['harga_satuan'] = df['harga_satuan'].astype(float)

print('Tipe data harga_satuan sekarang:', df['harga_satuan'].dtype)
df['harga_satuan'].describe()

Baris berformat titik-ribuan : 45
Baris berformat koma-desimal : 51
Tipe data harga_satuan sekarang: float64


,harga_satuan
count,"5,030.00"
mean,"22,185.09"
std,"15,205.70"
min,"4,000.00"
25%,"11,000.00"
50%,"16,000.00"
75%,"32,000.00"
max,"54,000.00"


## 1.2 Penanganan kuantitas negatif (`jumlah`)
Nilai negatif berasal dari retur/koreksi. Dengan Boolean Indexing + `.loc[]`, nilai `jumlah < 0` diganti menjadi nilai mutlaknya.

In [49]:
mask_neg = df['jumlah'] < 0
print('Jumlah baris bernilai negatif (sebelum):', mask_neg.sum())
print('Nilai jumlah minimum (sebelum)         :', df['jumlah'].min())

df.loc[mask_neg, 'jumlah'] = df.loc[mask_neg, 'jumlah'].abs()

print('Jumlah baris bernilai negatif (sesudah):', (df['jumlah'] < 0).sum())
print('Nilai jumlah minimum (sesudah)         :', df['jumlah'].min())

Jumlah baris bernilai negatif (sebelum): 41
Nilai jumlah minimum (sebelum)         : -12
Jumlah baris bernilai negatif (sesudah): 0
Nilai jumlah minimum (sesudah)         : 1


## 1.3 Imputasi nilai kosong `kode_kasir`

In [50]:
print('NaN kode_kasir (sebelum):', df['kode_kasir'].isna().sum())

df.loc[df['kode_kasir'].isna(), 'kode_kasir'] = 'K_UNKNOWN'

print('NaN kode_kasir (sesudah):', df['kode_kasir'].isna().sum())
print('Baris berlabel K_UNKNOWN :', (df['kode_kasir'] == 'K_UNKNOWN').sum())

NaN kode_kasir (sebelum): 261
NaN kode_kasir (sesudah): 0
Baris berlabel K_UNKNOWN : 261


## 1.4 Seleksi posisi dengan `.iloc[]`
Menampilkan 5 baris pertama untuk kolom `id_transaksi`, `tanggal`, `kode_gerai`, dan `jumlah`. Posisi kolom tersebut adalah 0, 1, 2, dan 5.

In [51]:
# Pastikan posisi kolom [0, 1, 2, 5] memang kolom yang diminta
print(df.columns[[0, 1, 2, 5]].tolist())

df.iloc[:5, [0, 1, 2, 5]]

['id_transaksi', 'tanggal', 'kode_gerai', 'jumlah']


,id_transaksi,tanggal,kode_gerai,jumlah
0,TRX0241065,2025-06-02,G09,4
1,TRX0240510,2024-12-28,G09,1
2,TRX0244483,2024-09-04,G11,3
3,TRX0242532,2025-07-30,G05,2
4,TRX0243509,26/03/2025,G05,1


---
# Task 2: Operasi Aritmatika (Feature Engineering)

Seluruh perhitungan dilakukan secara vektorisasi (tanpa perulangan).

| Kolom baru | Rumus |
|---|---|
| `Total_Kotor` | `harga_satuan * jumlah` |
| `Nominal_Diskon` | `Total_Kotor * (diskon_persen / 100)` |
| `Total_Bersih` | `(Total_Kotor - Nominal_Diskon) * 1.11` (PPN 11% setelah diskon) |

In [52]:
# 2.1 Omzet kotor
df['Total_Kotor'] = df['harga_satuan'] * df['jumlah']

# 2.2 Nominal diskon (diskon_persen berskala 0-100, jadi dibagi 100)
df['Nominal_Diskon'] = df['Total_Kotor'] * (df['diskon_persen'] / 100)

# 2.3 Omzet bersih = (kotor - diskon) + PPN 11%
df['Total_Bersih'] = (df['Total_Kotor'] - df['Nominal_Diskon']) * 1.11

df[['harga_satuan', 'jumlah', 'diskon_persen',
    'Total_Kotor', 'Nominal_Diskon', 'Total_Bersih']].head(10)

,harga_satuan,jumlah,diskon_persen,Total_Kotor,Nominal_Diskon,Total_Bersih
0,"13,000.00",4,0,"52,000.00",0.00,"57,720.00"
1,"21,000.00",1,0,"21,000.00",0.00,"23,310.00"
2,"7,500.00",3,0,"22,500.00",0.00,"24,975.00"
3,"6,500.00",2,0,"13,000.00",0.00,"14,430.00"
4,"6,000.00",1,0,"6,000.00",0.00,"6,660.00"
5,"45,500.00",12,10,"546,000.00","54,600.00","545,454.00"
6,"18,000.00",1,0,"18,000.00",0.00,"19,980.00"
7,"47,500.00",4,10,"190,000.00","19,000.00","189,810.00"
8,"11,000.00",3,0,"33,000.00",0.00,"36,630.00"
9,"6,500.00",2,0,"13,000.00",0.00,"14,430.00"


In [53]:
# Sanity check: contoh perhitungan manual untuk baris pertama
r = df.iloc[0]
manual = (r['harga_satuan'] * r['jumlah'] * (1 - r['diskon_persen'] / 100)) * 1.11
print('Total_Bersih (kolom) :', r['Total_Bersih'])
print('Total_Bersih (manual):', manual)

Total_Bersih (kolom) : 57720.00000000001
Total_Bersih (manual): 57720.00000000001


---
# Task 3: Filtering (Penyaringan Data)

## 3.1 Filter multi-kondisi
Syarat: `status == 'SELESAI'` **dan** gerai `G05` **atau** `G09` **dan** `Total_Bersih > 100.000`.
Setiap kondisi diberi kurung; kondisi `G05 | G09` dikurung lagi sebagai satu kelompok agar prioritas operator `&` dan `|` benar.

In [54]:
filter_3_1 = df[
    (df['status'] == 'SELESAI')
    & ((df['kode_gerai'] == 'G05') | (df['kode_gerai'] == 'G09'))
    & (df['Total_Bersih'] > 100_000)
]

print('Jumlah baris yang memenuhi kriteria:', len(filter_3_1))
filter_3_1.head(10)

Jumlah baris yang memenuhi kriteria: 140


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
7,TRX0241251,2024-10-09,G09,K035,SGR1204,4,"47,500.00",10,SELESAI,"190,000.00","19,000.00","189,810.00"
24,TRX0244253,2024-07-18,G09,K026,BYI1502,4,"30,500.00",0,SELESAI,"122,000.00",0.00,"135,420.00"
25,TRX0244999,2024-08-09,G05,K015,MIN1003,4,"28,500.00",0,SELESAI,"114,000.00",0.00,"126,540.00"
90,TRX0240898,2025-12-09,G05,K010,BYI1509,3,"46,500.00",0,SELESAI,"139,500.00",0.00,"154,845.00"
206,TRX0241868,2025-07-01,G05,K009,SGR1203,6,"20,500.00",0,SELESAI,"123,000.00",0.00,"136,530.00"
213,TRX0244651,2025-04-15,G09,K010,PRW1303,2,"46,000.00",0,SELESAI,"92,000.00",0.00,"102,120.00"
216,TRX0241198,2024-06-16,G09,K020,BYI1502,6,"28,000.00",0,SELESAI,"168,000.00",0.00,"186,480.00"
220,TRX0241165,2025-09-24,G09,K023,SGR1208,6,"27,500.00",0,SELESAI,"165,000.00",0.00,"183,150.00"
373,TRX0240601,2025-04-28,G05,K040,SGR1204,12,"50,000.00",0,SELESAI,"600,000.00",0.00,"666,000.00"
410,TRX0241526,2024-12-31,G05,K026,BYI1501,6,"18,500.00",0,SELESAI,"111,000.00",0.00,"123,210.00"


In [55]:
# Verifikasi silang dengan .isin()
cek = df[(df['status'] == 'SELESAI')
         & (df['kode_gerai'].isin(['G05', 'G09']))
         & (df['Total_Bersih'] > 100_000)]
print('Hasil sama dengan versi operator logika:', len(cek) == len(filter_3_1))

Hasil sama dengan versi operator logika: True


## 3.2 Filter dengan `.query()`
Transaksi gerai `G10` dengan jumlah pembelian minimal 5 unit.

In [56]:
filter_3_2 = df.query("kode_gerai == 'G10' and jumlah >= 5")

print('Jumlah baris:', len(filter_3_2))
filter_3_2.head(10)

Jumlah baris: 39


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
83,TRX0243880,2024-08-04,G10,K002,MKN1101,12,"45,500.00",0,SELESAI,"546,000.00",0.00,"606,060.00"
166,TRX0241592,2024-05-11,G10,K033,SGR1202,6,"13,000.00",0,SELESAI,"78,000.00",0.00,"86,580.00"
351,TRX0242686,2024-02-03,G10,K016,BYI1502,12,"26,000.00",0,SELESAI,"312,000.00",0.00,"346,320.00"
556,TRX0241396,2025-12-16,G10,K009,RMT1406,12,"5,500.00",5,SELESAI,"66,000.00","3,300.00","69,597.00"
879,TRX0243559,2025-09-30,G10,K016,SGR1206,6,"47,000.00",0,SELESAI,"282,000.00",0.00,"313,020.00"
964,TRX0244939,2025-07-08,G10,K009,SGR1200,6,"52,000.00",10,SELESAI,"312,000.00","31,200.00","311,688.00"
989,TRX0243076,2025-02-18,G10,K003,MKN1100,6,"18,000.00",0,SELESAI,"108,000.00",0.00,"119,880.00"
1023,TRX0242437,2025-09-18,G10,K030,MKN1104,6,"6,000.00",0,SELESAI,"36,000.00",0.00,"39,960.00"
1260,TRX0240537,2025-05-09,G10,K027,BYI1509,6,"54,000.00",0,SELESAI,"324,000.00",0.00,"359,640.00"
1296,TRX0240698,2024-01-24,G10,K022,MKN1109,6,"29,000.00",0,SELESAI,"174,000.00",0.00,"193,140.00"


---
# Task 4: Sorting (Pengurutan Data)

## 4.1 Multi-column sorting
`kode_gerai` A–Z (ascending), lalu `Total_Bersih` dari tertinggi ke terendah (descending).

In [57]:
df_urut = df.sort_values(by=['kode_gerai', 'Total_Bersih'], ascending=[True, False])

df_urut[['id_transaksi', 'kode_gerai', 'status', 'jumlah', 'Total_Bersih']].head(10)

,id_transaksi,kode_gerai,status,jumlah,Total_Bersih
244,TRX0243257,G01,SELESAI,1456,"42,020,160.00"
2956,TRX0241367,G01,SELESAI,12,"672,660.00"
3588,TRX0240608,G01,SELESAI,12,"626,040.00"
2937,TRX0240631,G01,SELESAI,12,"559,440.00"
5,TRX0240658,G01,SELESAI,12,"545,454.00"
3889,TRX0241537,G01,SELESAI,12,"359,640.00"
2179,TRX0244635,G01,SELESAI,6,"349,650.00"
4270,TRX0240428,G01,SELESAI,6,"323,010.00"
695,TRX0244706,G01,BATAL,6,"313,020.00"
4404,TRX0242179,G01,SELESAI,6,"313,020.00"


## 4.2 Top-N dengan `.nlargest()`
5 transaksi dengan `Total_Bersih` tertinggi di seluruh dataset.

In [58]:
top_5 = df.nlargest(5, 'Total_Bersih')

top_5[['id_transaksi', 'tanggal', 'kode_gerai', 'status', 'jumlah',
       'harga_satuan', 'diskon_persen', 'Total_Bersih']]

,id_transaksi,tanggal,kode_gerai,status,jumlah,harga_satuan,diskon_persen,Total_Bersih
994,TRX0241002,2024-03-18,G11,SELESAI,2093,"48,000.00",10,"100,363,536.00"
4985,TRX0244221,2024-12-09,G09,SELESAI,1301,"51,500.00",10,"66,934,498.50"
467,TRX0243696,2024-02-27,G06,SELESAI,892,"54,000.00",0,"53,466,480.00"
3393,TRX0241029,2025-05-12,G07,SELESAI,1983,"23,000.00",10,"45,563,391.00"
244,TRX0243257,2024-11-24,G01,SELESAI,1456,"32,500.00",20,"42,020,160.00"


---
# Task 5: Grouping & Agregasi (Pelaporan Kinerja Gerai)

## 5.1 Laporan agregasi per gerai
Data disaring dulu hanya untuk transaksi `SELESAI`, lalu diringkas per `kode_gerai` dengan *named aggregation*.

In [59]:
df_selesai = df[df['status'] == 'SELESAI']

laporan_gerai = (
    df_selesai
    .groupby('kode_gerai')
    .agg(
        Jumlah_Transaksi=('id_transaksi', 'count'),
        Total_Unit_Terjual=('jumlah', 'sum'),
        Total_Omzet_Bersih=('Total_Bersih', 'sum'),
        Rata_Rata_Transaksi=('Total_Bersih', 'mean'),
    )
    .sort_values('Total_Omzet_Bersih', ascending=False)
    .round(2)
)

laporan_gerai

,Jumlah_Transaksi,Total_Unit_Terjual,Total_Omzet_Bersih,Rata_Rata_Transaksi
kode_gerai,,,,
G11,398,3170,"125,638,152.75","315,673.75"
G09,379,3027,"101,157,630.00","266,906.68"
G06,383,3103,"91,228,624.50","238,194.84"
G07,384,4284,"80,559,748.50","209,791.01"
G01,450,2635,"70,967,378.25","157,705.28"
G03,372,3274,"59,991,226.50","161,266.74"
G10,401,1742,"58,964,892.75","147,044.62"
G08,408,1700,"35,630,084.25","87,328.64"
G12,406,1010,"24,699,220.50","60,835.52"


## 5.2 Transformasi kelompok (window function)
`.groupby().transform('sum')` menghasilkan total omzet gerai untuk **setiap baris** (panjangnya sama dengan DataFrame utama), sehingga bisa langsung dibagi dengan `Total_Bersih` tiap transaksi.

$$\text{Kontribusi_Gerai_Persen} = \frac{\text{Total_Bersih}}{\text{Total omzet bersih gerai tersebut}} \times 100$$

Kolom ini ditambahkan ke DataFrame utama `df` (semua status transaksi), sesuai bunyi soal.

In [60]:
total_omzet_gerai = df.groupby('kode_gerai')['Total_Bersih'].transform('sum')

df['Kontribusi_Gerai_Persen'] = (df['Total_Bersih'] / total_omzet_gerai) * 100

df[['id_transaksi', 'kode_gerai', 'Total_Bersih', 'Kontribusi_Gerai_Persen']].head(10).sort_values('Kontribusi_Gerai_Persen', ascending=False)

,id_transaksi,kode_gerai,Total_Bersih,Kontribusi_Gerai_Persen
5,TRX0240658,G01,"545,454.00",0.75
7,TRX0241251,G09,"189,810.00",0.19
3,TRX0242532,G05,"14,430.00",0.06
0,TRX0241065,G09,"57,720.00",0.06
8,TRX0244274,G07,"36,630.00",0.04
9,TRX0243163,G08,"14,430.00",0.04
6,TRX0244581,G10,"19,980.00",0.03
4,TRX0243509,G05,"6,660.00",0.03
1,TRX0240510,G09,"23,310.00",0.02
2,TRX0244483,G11,"24,975.00",0.02


In [61]:
# Validasi: kontribusi seluruh transaksi dalam satu gerai harus berjumlah 100%
df.groupby('kode_gerai')['Kontribusi_Gerai_Persen'].sum().round(6)

,Kontribusi_Gerai_Persen
kode_gerai,
G01,100.00
G02,100.00
G03,100.00
G04,100.00
G05,100.00
G06,100.00
G07,100.00
G08,100.00
G09,100.00
